# MACS 30111 — M7 Pandas, day 3: pivot, then merge
Slides: `M7_Pandas_30111.pptx`, slides 45–58 · Textbook: §4.3.6 ("Pivoting") · Docs: https://pandas.pydata.org/docs/user_guide/merging.html

**How to use this notebook:** run the **Example** cells as we go. In each **Try it** cell, predict the answer first, then write the code. Try its marked **(on your own)** are homework.

*`read_excel` needs the `openpyxl` package. If loading the lifespan file gives an `ImportError`, run `pip install openpyxl` once, then restart the kernel.*

In [ ]:
import pandas as pd

trees = pd.read_csv("data/2015_Street_Tree_Census_sample.csv", index_col="tree_id")
trees.shape

## 1. Pivot, with and without `reset_index` (slides 45–52)
`pivot` turns a **long** table into a **wide** one. It needs three **columns**: one for the new row labels (`index=`), one for the new column names (`columns=`), and one for the values (`values=`).

Start with a count of trees for each borough and status. **Predict:** is `counts` a Series or a DataFrame? Where do borough and status live?

In [ ]:
counts = trees.groupby(["borough", "status"]).size()
print(type(counts))
counts.head()

### Example: pivot *without* `reset_index`
**Predict:** will this work? (This was quiz question 6.)

In [ ]:
counts.pivot(index="borough", columns="status")

It raises an error. `counts` is a **Series**, and borough and status are its **index** (the row labels), not columns. A Series has no `pivot`, and even if it did, there would be no `borough` *column* to name.

### Example: pivot *with* `reset_index`
`reset_index()` moves the index levels into ordinary columns and gives the rows plain labels 0, 1, 2, ... Use `name=` to name the column that holds the values.

In [ ]:
long = counts.reset_index(name="n")
print(long.shape)       # (15, 3): one row per (borough, status)
long.head()

In [ ]:
wide = long.pivot(index="borough", columns="status", values="n")
print(wide.shape)       # (5, 3)
wide

**Compare:** `counts.unstack()` gives the same wide table in one step, because it works on the index directly. `pivot` works on columns, which is why it needs `reset_index` first.

What if you leave out `name=`? The new column is named `0`, so you have to rename it (the textbook does this):

In [ ]:
counts.reset_index().columns    # ['borough', 'status', 0]

`reset_index` works on a **DataFrame** too: it moves the index (here, `tree_id`) back into a column. Remember this for merging: a merge keeps columns, not the index.

In [ ]:
trees.reset_index()[["tree_id", "borough", "spc_common"]].head(3)

### Try it 1.1
Make a wide table of **health** counts: one row per borough, one column per health value (Fair, Good, Poor). Use `groupby` → `size()` → `reset_index(name="n")` → `pivot`. What is its shape?

In [ ]:
# your code here


### Try it 1.2 (on your own)
Make a wide table of the **average** `tree_dbh` for each borough (rows) and status (columns), this time starting from `.mean()` instead of `.size()`. Which `reset_index` call do you need? *(Hint: after `groupby(...)["tree_dbh"].mean()` the values already have a name.)*

In [ ]:
# your code here


## 2. Combining DataFrames (slide 53)
| Tool | What it does | Matches rows on |
|---|---|---|
| `pd.concat([df1, df2])` | stacks tables on top of each other | nothing: rows are just stacked |
| `df1.merge(df2)` | lines up rows whose key values match | key column(s) |
| `df1.join(df2)` | a merge that uses `df2`'s index as the key | `df2`'s index |

**Rule of thumb:** concat to add more rows of the same kind; merge to add columns from a different table.

In [ ]:
queens = trees[trees["borough"] == "Queens"]
bronx = trees[trees["borough"] == "Bronx"]
both_boroughs = pd.concat([queens, bronx])
print(len(queens), len(bronx), both_boroughs.shape)   # 36,479 + 12,596 = 49,075 rows

## 3. Merge: syntax and which rows are kept (slides 54–55)
```python
left.merge(right, how="inner", on=None, left_on=None, right_on=None,
           suffixes=("_x", "_y"), indicator=False)
```
`pd.merge(left, right, ...)` does exactly the same thing.

Two small tables, both with a `name` column:

In [ ]:
grades = pd.DataFrame({"name": ["Wei", "Priya", "Mateo"], "pa1": [92, 85, 78]})
majors = pd.DataFrame({"name": ["Priya", "Mateo", "Amara"],
                       "major": ["Econ", "Sociology", "CS"]})
print(grades)
print(majors)

**Predict before each cell:** which names will be in the result?

In [ ]:
grades.merge(majors, on="name", how="inner")    # the default

In [ ]:
grades.merge(majors, on="name", how="left")

In [ ]:
grades.merge(majors, on="name", how="right")

In [ ]:
grades.merge(majors, on="name", how="outer", indicator=True)

Notice:
- A missing match becomes `NaN`, and that turns the int column `pa1` into floats (85 → 85.0).
- `indicator=True` adds a `_merge` column: `left_only`, `right_only` or `both`.
- `pd.merge(grades, majors, on="name")` gives the same result as `grades.merge(majors, on="name")`.

### Try it 3.1
Without running anything first, write down how many **rows** each `how=` gives (inner, left, right, outer). Then check with a loop that prints `len(...)` for each one.

In [ ]:
# your code here


### Try it 3.2 (on your own)
Both tables now also have a `term` column. Run the merge two ways: with `on="name"`, and with no `on=` at all. How many rows does each give, and what happened to the `term` columns? Why is leaving out `on=` risky?

In [ ]:
# your code here


## 4. Example: add lifespans to the trees (slide 56)
`trees_lifespan.xlsx` has one row per **species**: its common name, scientific name, and average and maximum lifespan in years.

In [ ]:
trees_life = pd.read_excel("data/trees_lifespan.xlsx")
print(trees_life.shape)
trees_life.head()

**Which column should we match on?** Compare how the two tables write a red maple:

In [ ]:
print(trees.loc[180683, ["spc_common", "spc_latin"]])
print(trees_life[trees_life["Scientific Name"] == "Acer rubrum"])

In [ ]:
# matching on the common names: "red maple" vs "Maple, Red" never line up
trees.merge(trees_life, left_on="spc_common", right_on="Common Name").shape    # only 2 trees!

The scientific names are written the same way, so we match on those. The columns have **different names**, so we use `left_on` / `right_on`. `how="left"` keeps every tree, even if its species has no lifespan.

One more thing: **merge returns plain row labels 0, 1, 2, ...**, so the `tree_id` index would be lost. Use `reset_index()` first to keep it as a column.

In [ ]:
trees_plus = trees.reset_index().merge(trees_life, how="left",
                                       left_on="spc_latin", right_on="Scientific Name",
                                       indicator=True)
print(trees_plus.shape)     # (100000, 50): every tree kept
trees_plus[["tree_id", "spc_common", "Common Name", "Average_lifespan", "_merge"]].head()

### Try it 4.1
Run the same merge **without** `reset_index()`. What happened to the tree ids? How would you look up tree 180683 now?

In [ ]:
# your code here


## 5. Always check your merge (slide 57)
Compare the shape before and after, and look at the `_merge` counts.

In [ ]:
trees_plus["_merge"].value_counts()      # left_only 80,884 · both 19,116

### Try it 5.1
Most trees found no lifespan. Which **5 species** (by `spc_latin`) have the most unmatched trees?

In [ ]:
# your code here


### Try it 5.2 (on your own)
The lifespan file spells green ash *Fraxinus pennsylvanicum*; the census spells it *Fraxinus pennsylvanica*. Fix the spelling in `trees_life` with `.replace`, merge again, and count the matched trees. How many more trees found a lifespan?

In [ ]:
# your code here


## 6. TASK: ask a question with the merged data (slide 58)

### Try it 6.1
Which borough's street trees have the **longest average lifespan**? *(Hint: `groupby` after the merge. What happens to the trees with no lifespan?)*

In [ ]:
# your code here


### Try it 6.2 (on your own)
Among the trees that matched, which **species** is planted most often, and which matched species has the **longest** average lifespan?

In [ ]:
# your code here


## Next time
Plotting. Homework: the **on your own** Try its (1.2, 3.2, 5.2, 6.2).